# Calibration

STIsim provides a calibration framework built on [Starsim's calibration tools](https://docs.starsim.org) and [Optuna](https://optuna.org). If you're new to calibration, start with the [Starsim calibration tutorial](https://docs.starsim.org/en/latest/user_guide/workflows_calibration.html) which covers the fundamentals: parameter bounds, evaluation functions, calibration components, and Optuna diagnostics.

This tutorial covers what STIsim adds:
- **Dot-notation parameters** that route automatically to the right module
- **Multi-disease calibration** with joint HIV + STI fitting
- **`make_calib_sims()`** for running calibrated parameter sets at scale

## Setup: generate synthetic data

To demonstrate calibration without external data files, we'll generate synthetic targets from a simulation with known "true" parameters. In practice, you'd load survey data (e.g., DHS, PHIA) instead.

In [ ]:
import numpy as np
import pandas as pd
import sciris as sc
import starsim as ss
import stisim as sti

# "True" parameters we'll try to recover
true_beta = 0.08
true_condom = 0.6

# Create and run a sim with the true parameters
true_sim = sti.Sim(
    diseases=sti.Gonorrhea(beta_m2f=true_beta, eff_condom=true_condom),
    n_agents=2000, start=2010, stop=2030,
)
true_sim.run(verbose=0)

# Extract yearly prevalence and incidence as our "data"
df = true_sim.to_df(resample='year', use_years=True, sep='.')
data = df[['timevec', 'ng.prevalence', 'ng.new_infections']].dropna()
data = data.rename(columns={'timevec': 'time'})
data['time'] = data['time'].astype(int)
print(f'Generated {len(data)} data points')
data.head()

## Defining calibration parameters

STIsim uses **dot notation** to route parameters to the right module: `'module_name.par_name'`. The module name is whatever `sim.get_module()` would find -- typically the disease name (`ng`, `hiv`, `syph`), network name (`structuredsexual`), connector name (`hiv_syph`), or intervention name (`art`, `fsw_testing`).

You can write parameters in two equivalent formats:

In [ ]:
# Nested format -- group parameters by module (Pythonic, uses keyword syntax)
calib_pars = dict(
    ng=dict(
        beta_m2f=  dict(low=0.03, high=0.15, guess=0.06),
        eff_condom=dict(low=0.3,  high=0.9,  guess=0.5),
    ),
)

# Flat format -- equivalent, uses string keys
calib_pars_flat = {
    'ng.beta_m2f':   dict(low=0.03, high=0.15, guess=0.06),
    'ng.eff_condom':  dict(low=0.3,  high=0.9,  guess=0.5),
}

# Both produce the same result after flattening
print(sti.flatten_calib_pars(calib_pars))
print(calib_pars_flat)

Each parameter spec requires `low` and `high` bounds. The optional `guess` is used as the starting point for the "before" comparison in `check_fit()`. 

The nested format is particularly convenient when calibrating multiple modules -- parameters are visually grouped:

In [ ]:
# Multi-module example (not run here -- just to show the pattern)
multi_pars = dict(
    hiv=dict(
        beta_m2f=dict(low=0.002, high=0.014, guess=0.006),
        eff_condom=dict(low=0.5, high=0.9, guess=0.75),
    ),
    structuredsexual=dict(
        prop_f0=dict(low=0.55, high=0.9, guess=0.7),
        m1_conc=dict(low=0.05, high=0.3, guess=0.15),
    ),
    syph=dict(
        beta_m2f=dict(low=0.15, high=0.35, guess=0.2),
    ),
)
print(f'{len(sti.flatten_calib_pars(multi_pars))} parameters across 3 modules')

## Running a calibration

Create a `Calibration` with the sim, parameters, and data. No custom `build_fn` is needed -- STIsim's default automatically routes parameters using dot notation.

In [ ]:
sim = sti.Sim(diseases='ng', n_agents=500, start=2010, stop=2030)

calib = sti.Calibration(
    sim=sim,
    calib_pars=calib_pars,
    data=data,
    total_trials=10,
    n_workers=1,
)
calib.calibrate()
print(f'Best parameters: {calib.best_pars}')

The calibration found parameters that minimize the mismatch between model output and data. Let's see how they compare to the true values:

In [ ]:
true_pars = {'ng.beta_m2f': true_beta, 'ng.eff_condom': true_condom}
for par, true_val in true_pars.items():
    best_val = calib.best_pars[par]
    print(f'{par}: true={true_val:.3f}, calibrated={best_val:.3f}')

With only 10 trials and 500 agents, recovery won't be perfect. In practice, you'd use 1000-2000 trials with 5000-10000 agents.

## Extracting calibrated parameters

After calibration, use `get_pars()` to extract the top parameter sets as flat dicts ready to feed back into a sim:

In [ ]:
# Get top 3 parameter sets (sorted by mismatch)
par_sets = calib.get_pars(n=3)
for i, pars in enumerate(par_sets):
    print(f'Set {i}: {pars}')

You can also access the full results DataFrame via `calib.df`, which includes all parameters plus the mismatch value for every trial.

## Running calibrated sims with `make_calib_sims`

The most common post-calibration task is running the top N parameter sets to generate results with uncertainty. `make_calib_sims()` handles this in one call:

In [ ]:
# Run top 3 parameter sets
msim = sti.make_calib_sims(calib=calib, n_parsets=3)
print(f'Ran {len(msim.sims)} simulations')
print(f'Each sim has par_idx: {[s.par_idx for s in msim.sims]}')

You can also pass calibration parameters directly -- useful when loading saved results:

In [ ]:
# From a list of parameter dicts
msim = sti.make_calib_sims(
    calib_pars=par_sets,
    sim=sti.Sim(diseases='ng', n_agents=500, start=2010, stop=2030),
)

# From a DataFrame (like calib.df)
msim = sti.make_calib_sims(
    calib_pars=calib.df,
    sim=sti.Sim(diseases='ng', n_agents=500, start=2010, stop=2030),
    n_parsets=3,
)

### Filtering with `check_fn`

Some parameter combinations may produce epidemiologically implausible results (e.g., disease die-out). Pass a `check_fn` to filter:

In [ ]:
def check_ng_alive(sim):
    """Reject sims where gonorrhea died out."""
    return float(sim.results.ng.new_infections[-12:].sum()) > 0

msim = sti.make_calib_sims(
    calib=calib, n_parsets=3,
    check_fn=check_ng_alive,
)
print(f'Kept {len(msim.sims)} sims after filtering')

### Multiple seeds per parameter set

For stochastic robustness, run each parameter set with multiple random seeds. When combined with `check_fn`, only the first surviving seed per parameter set is kept:

In [ ]:
msim = sti.make_calib_sims(
    calib=calib, n_parsets=3,
    seeds_per_par=2,
    check_fn=check_ng_alive,
)
print(f'Kept {len(msim.sims)} sims (1 per par set, best surviving seed)')

## Saving and loading

After calibration, use `calib.save()` to save the calibration object and parameter DataFrame. This handles shrinking (keeping only the top results to reduce file size) automatically:

In [ ]:
import tempfile, os
tmpdir = tempfile.mkdtemp()

# Save with shrinking (keeps top 10% by default)
calib.save(os.path.join(tmpdir, 'my_calib.obj'))

# Load back and use
loaded = sc.load(os.path.join(tmpdir, 'my_calib.obj'))
print(f'Loaded calibration with {len(loaded.df)} parameter sets')

## Preparing real data

The calibration expects a DataFrame with a `time` column (integer years) and columns matching simulation result names. For example:

| time | hiv.prevalence | syph.active_prevalence | hiv.n_on_art |
|------|---------------|----------------------|-------------|
| 2010 | 0.12 | 0.03 | 50000 |
| 2015 | 0.11 | 0.025 | 120000 |
| 2020 | 0.098 | 0.02 | 180000 |

Column names must match the result names in the simulation (use `sim.to_df().columns` to see available names). Missing years are fine -- the calibration will only compare at timepoints where data exists.

You can also provide `weights` to emphasize certain targets:

In [ ]:
# Example weights (not run -- just showing the pattern)
weights = {
    'hiv.prevalence': 5.0,      # Emphasize HIV prevalence
    'syph.active_prevalence': 10.0,  # Even more weight on syphilis
    'hiv.n_on_art': 2.0,
}

## Typical production workflow

A complete calibration analysis typically has three scripts:

```python
# 1. run_calibrations.py -- find best parameters
sim = make_sim(verbose=-1)
data = pd.read_csv('data/calibration_targets.csv')

calib = sti.Calibration(
    sim=sim,
    calib_pars=dict(
        hiv=dict(beta_m2f=dict(low=0.002, high=0.014, guess=0.006)),
        structuredsexual=dict(prop_f0=dict(low=0.55, high=0.9, guess=0.7)),
    ),
    data=data,
    weights={'hiv.prevalence': 5.0},
    total_trials=2000,
)
calib.calibrate()
calib.save('results/calib.obj')
```

```python
# 2. run_msim.py -- run top parameter sets with full results
pars_df = sc.load('results/calib_pars.df')
msim = sti.make_calib_sims(
    calib_pars=pars_df, sim=make_sim(), n_parsets=200,
)
```

```python
# 3. run_scenarios.py -- compare interventions
for scenario in ['baseline', 'intervention']:
    msim = sti.make_calib_sims(
        calib_pars=pars_df, sim=make_sim(scenario=scenario),
        n_parsets=10, seeds_per_par=5,
    )
```

## Further reading

- **[Starsim calibration tutorial](https://docs.starsim.org)**: Calibration components, Bayesian sampling, Optuna diagnostics
- **[Optuna documentation](https://optuna.org)**: Pruning, samplers, distributed optimization